# Task 3 — Usage E8 with 687 added images

**This follows the current main EDA's saved teacher folds.** The dataset keeps the earlier 120 additions and adds 567 more images. All earlier IDs and folds stay fixed.

**Run All trains five fresh models, one per fold, for 30 epochs each.** The model recipe stays the same as E8. This tests the effect of adding data.

| Control | Fixed value |
|---|---|
| Target | Usage; the original nine classes, including NA |
| Input | RGB, 60 pixels wide × 80 pixels high |
| Model | SmallCNN; widths 32, 64, 128, 256; average pooling |
| Epochs / batch size | 30 / 128 |
| Optimizer | AdamW; learning rate 0.001; weight decay 0.0001 |
| Schedule | Cosine; final learning rate 0.00001 |
| Loss | Weighted cross-entropy; effective-number beta 0.999, cap 5 |
| Augmentation | Image shifts of up to 2 pixels, with probability 0.5 |
| Seed / precision | 2753 / FP32 |
| Checkpoint | Final epoch; fresh weights; no early stopping |

The main comparison uses the **same teacher validation images** for three models: original E8, E8 with 120 added images, and this version with 687. Added-source scores are shown separately.


## 1. Start in Colab

1. Select a **GPU** in **Runtime → Change runtime type**.
2. Put `teacher_plus_rare_usage_v2_training.zip` in `MyDrive/MLA2/data/`.
3. Keep the existing `task3-data.zip` in that same folder.
4. Choose **Run All**.

The new training ZIP includes the code, splits, all 687 added images and both saved references. You do not need the separate dataset-only ZIP. Results go to `MyDrive/MLA2/task3_usage_expanded_v2_e8/`.

After a disconnect, use Run All again. Complete folds are checked and reused. An unfinished fold starts again with fresh weights.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import tempfile
import zipfile

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive/MLA2"
BUNDLE_ZIP = DRIVE_PROJECT / "data/teacher_plus_rare_usage_v2_training.zip"
TEACHER_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3_usage_expanded_v2_e8"


def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def safe_members(archive, prefix=None):
    members = archive.infolist()
    names = [member.filename for member in members]
    if len(names) != len(set(names)):
        raise ValueError("Archive contains duplicate file names.")
    for member in members:
        name = member.filename
        if Path(name).is_absolute() or ".." in Path(name).parts or "\\" in name:
            raise ValueError(f"Unsafe archive path: {name}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f"Archive symlink is not allowed: {name}")
    return [m for m in members if prefix is None or m.filename.startswith(prefix)]


if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(f"Upload the new training ZIP here first: {BUNDLE_ZIP}")
bundle_digest = sha256_file(BUNDLE_ZIP)
LOCAL_BUNDLE = Path("/content") / f"usage-v2-training-{bundle_digest[:12]}.zip"
if not LOCAL_BUNDLE.is_file() or sha256_file(LOCAL_BUNDLE) != bundle_digest:
    partial = LOCAL_BUNDLE.with_suffix(".zip.partial")
    shutil.copyfile(BUNDLE_ZIP, partial)
    if sha256_file(partial) != bundle_digest:
        raise RuntimeError("Training ZIP copy is incomplete. Run this cell again.")
    partial.replace(LOCAL_BUNDLE)
REPO_DIR = Path("/content") / f"MLA2-usage-expanded-v2-{bundle_digest[:12]}"
with zipfile.ZipFile(LOCAL_BUNDLE) as archive:
    members = safe_members(archive)
    manifest = json.loads(archive.read("training_bundle_manifest.json"))
    if manifest["dataset"]["name"] != "teacher_plus_rare_usage_v2_20260906":
        raise ValueError("This notebook requires the new v2 training bundle.")
    expected = manifest["files"]
    actual = {m.filename for m in members if not m.is_dir()}
    if actual != set(expected) | {"training_bundle_manifest.json"}:
        raise ValueError("The training archive inventory differs from its manifest.")
    for name, digest in expected.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != digest:
            raise ValueError(f"Training archive file changed: {name}")
    if not REPO_DIR.exists():
        with tempfile.TemporaryDirectory(prefix="usage-v2-unpack-", dir="/content") as temporary:
            staging = Path(temporary) / "project"
            staging.mkdir()
            archive.extractall(staging)
            for name, digest in expected.items():
                if sha256_file(staging / name) != digest:
                    raise RuntimeError(f"Extracted bundle file differs: {name}")
            staging.rename(REPO_DIR)
for name, digest in expected.items():
    if not (REPO_DIR / name).is_file() or sha256_file(REPO_DIR / name) != digest:
        raise RuntimeError(f"Local bundle file differs: {name}. Use a fresh runtime.")

if "fashion.config" in sys.modules:
    loaded_root = Path(sys.modules["fashion.config"].ROOT)
    if loaded_root != REPO_DIR:
        raise RuntimeError("Another notebook is loaded. Restart the session, then Run All.")
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
REFERENCE_ROOT = REPO_DIR / "reference/usage_expanded_v2"
print("New Usage code and data ready:", REPO_DIR)
print("Bundle SHA-256:", bundle_digest)


## 2. Load the teacher images

The dataset contains **39,299 rows**: 38,612 teacher rows plus 687 additions. Training and validation use **33,459 rows with Usage labels**. The other rows are the saved holdout, quarantined rows, and one teacher row without a Usage label.

The five folds use the same main-EDA teacher assignments. New images stay with their related product groups. This notebook uses development data only.


In [ ]:
if not TEACHER_ZIP.is_file():
    raise FileNotFoundError(f"Existing teacher archive is missing: {TEACHER_ZIP}")
LOCAL_TEACHER_ZIP = Path("/content/task3-data.zip")
if (
    not LOCAL_TEACHER_ZIP.is_file()
    or LOCAL_TEACHER_ZIP.stat().st_size != TEACHER_ZIP.stat().st_size
):
    partial = LOCAL_TEACHER_ZIP.with_suffix(".zip.partial")
    shutil.copyfile(TEACHER_ZIP, partial)
    if partial.stat().st_size != TEACHER_ZIP.stat().st_size:
        raise RuntimeError("Teacher ZIP copy is incomplete. Run this cell again.")
    partial.replace(LOCAL_TEACHER_ZIP)
with zipfile.ZipFile(LOCAL_TEACHER_ZIP) as archive:
    members = safe_members(archive, prefix="data/raw/teacher/")
    if not members:
        raise ValueError("Teacher archive must contain data/raw/teacher/.")
    for member in members:
        destination = REPO_DIR / member.filename
        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        elif not destination.is_file() or destination.stat().st_size != member.file_size:
            archive.extract(member, REPO_DIR)
print("Teacher images are on local disk.")


## 3. Check the selected dataset

This checks the new split, class map, earlier fold assignments and source-border geometry. Training checks every image it uses before the first fold.

Channel means, channel spreads and class weights are calculated from each fold's **training images only**. The saved borders exclude added white padding from channel statistics.

Use the normal Colab GPU runtime. The trainer records the actual library versions.


In [ ]:
import pandas as pd
import torch
from IPython.display import Image as DisplayImage, display
from fashion.train.task3_usage_expanded_v2 import (
    expanded_usage_spec,
    run_expanded_usage_v2,
    save_learning_curves,
    usage_registry_path,
    validate_dataset,
)

if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime, then Run All.")
splits, contract = validate_dataset(root=REPO_DIR, check_images=False)
DRIVE_REGISTRY = usage_registry_path(DRIVE_TASK_DIR)
E8_DIR = REFERENCE_ROOT / "teacher_e8"
SOURCE_REGISTRY = REFERENCE_ROOT / "teacher_e8_runs.csv"
PREVIOUS_DIR = REFERENCE_ROOT / "previous_expansion"
PREVIOUS_REGISTRY = REFERENCE_ROOT / "previous_expansion_runs.csv"

print("Dataset:", contract["name"])
print("Usage development images:", contract["usage_development_rows"])
print("GPU:", torch.cuda.get_device_name(0))
print("Results:", DRIVE_TASK_DIR)
print("Run log:", DRIVE_REGISTRY)
counts = pd.DataFrame(contract["folds"])
display(counts[[
    "fold", "training_rows", "validation_rows",
    "new_added_training_rows", "new_added_validation_rows",
]].rename(columns={
    "fold": "Fold", "training_rows": "Train", "validation_rows": "Validation",
    "new_added_training_rows": "New in train",
    "new_added_validation_rows": "New in validation",
}))
print("Recipe:", expanded_usage_spec().name)


## 4. Train all five folds

Each fold starts from scratch and runs 30 epochs. The saved E8 and first-expansion models are read only to verify reference results. Their weights are never copied into the new models.

Each run is logged before its first training step. The notebook saves checkpoints, epoch histories, clean training predictions, validation predictions, source-specific scores and corruption checks. A separate v2 run log keeps these results apart from the earlier runs.

Keep all five folds for the full comparison. A smaller fold list is saved as a partial result and is compared only against the matching reference folds.


In [ ]:
FOLDS = (0, 1, 2, 3, 4)
result = run_expanded_usage_v2(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
    e8_directory=E8_DIR,
    source_registry_path=SOURCE_REGISTRY,
    previous_directory=PREVIOUS_DIR,
    previous_registry_path=PREVIOUS_REGISTRY,
    folds=FOLDS,
    resume=True,
)
print("Saved teacher-only comparison:", result["comparison_path"])


## 5. Read the results

**Use the teacher-only comparison to judge whether this data helps the assignment task.** All three models are scored on the same teacher images with the same nine-class map.

The combined and added-source scores help explain what the model learned. Their class mix is different, so a higher combined score alone does not show better teacher-test performance.

There are 136 added Home, 224 Party, 162 Smart Casual and 165 Travel images across the two intakes. No NA images were added. The new labels come from product text or retailer collections. A 49-image Smart Casual family makes the new fold sizes uneven.

Earlier holdout and teacher-test evaluations already exist. This notebook keeps fitting and comparison within development data.


In [ ]:
summary = result["comparison"]
teacher = summary["sources"]["teacher"]["metrics"]
reference_names = {
    "teacher_e8": "Original E8",
    "previous_expansion": "E8 + 120 added images",
}
scores = [
    {"Model": reference_names[name], "Teacher macro-F1": metrics["macro_f1"]}
    for name, metrics in summary["teacher_references"].items()
]
scores.append({"Model": "E8 + 687 added images", "Teacher macro-F1": teacher["macro_f1"]})
display(pd.DataFrame(scores))
print("Same teacher images in each comparison:", summary["sources"]["teacher"]["rows"])
display(pd.DataFrame(summary["teacher_per_class"]))
display(pd.DataFrame([
    {"Source": name, "Images": scope["rows"],
     "Macro-F1": scope["metrics"]["macro_f1"] if scope["metrics"] else None}
    for name, scope in summary["sources"].items()
]))
display(pd.DataFrame([
    {
        "Fold": run["metrics"]["validation_fold"],
        "Teacher train F1": run["metrics"]["source_metrics"]["clean_training"]["teacher"]["metrics"]["macro_f1"],
        "Teacher validation F1": run["metrics"]["source_metrics"]["validation"]["teacher"]["metrics"]["macro_f1"],
    }
    for run in result["fold_results"]
]))
print("Result files:", Path(result["comparison_path"]).parent)
print("Run registry:", result["registry_path"])


## 6. Check learning across epochs

These curves show validation on the combined teacher and external data. The teacher-only table above remains the main comparison. The model uses epoch 30; these curves do not select a different checkpoint.


In [ ]:
curve_path = save_learning_curves(
    result,
    path=DRIVE_TASK_DIR / "results/figures/task3/usage_expanded_v2_e8_learning_curves.png",
)
display(DisplayImage(filename=str(curve_path)))
print("Saved curves:", curve_path)
